In [1]:
from importlib import reload
import utils
reload(utils)
from utils import *
execute_smoke_tests()

drawdown smoke test passed
sharpe smoke test passed
sortino smoke test passed
limit detection smoke test: OK
All smoke tests passed.


In [2]:
import numpy as np
import pandas as pd
from utils import compute_sharpe, TRADING_DAYS_A_SHARE

# ---- 1. Build / reuse index_returns ----
# If you still have a dict of {label: returns_series} for the 4 indices in memory
# from Session 2, skip this loading block. Otherwise rebuild from the cache:

index_files = {
    '上证50':   'data/prices/sh_000016_2023-04-17_2026-04-17_qfq.csv',
    '沪深300':  'data/prices/sh_000300_2023-04-17_2026-04-17_qfq.csv',
    '中证1000': 'data/prices/sh_000852_2023-04-17_2026-04-17_qfq.csv',
    '创业板指': 'data/prices/sz_399006_2023-04-17_2026-04-17_qfq.csv',
}

index_returns = {}
for label, path in index_files.items():
    df = pd.read_csv(path, parse_dates=['date']).set_index('date').sort_index()
    close = pd.to_numeric(df['close'], errors='coerce')
    returns = close.pct_change().dropna()
    index_returns[label] = returns
    print(f"{label:10s}: {len(returns):4d} rows, "
          f"{returns.index.min().date()} → {returns.index.max().date()}")

上证50      :  726 rows, 2023-04-18 → 2026-04-17
沪深300     :  726 rows, 2023-04-18 → 2026-04-17
中证1000    :  726 rows, 2023-04-18 → 2026-04-17
创业板指      :  726 rows, 2023-04-18 → 2026-04-17


In [3]:
# ---- 2. Sharpe and its components ----
rows = []
for label, returns in index_returns.items():
    ann_mean = returns.mean() * TRADING_DAYS_A_SHARE
    ann_std  = returns.std()  * np.sqrt(TRADING_DAYS_A_SHARE)
    sharpe   = compute_sharpe(returns, rf_annual=0.0)
    rows.append({
        'Index': label,
        'Ann_Mean':   ann_mean,
        'Ann_Std':    ann_std,
        'Sharpe':     sharpe,
    })

sharpe_table = (
    pd.DataFrame(rows)
      .set_index('Index')
      .sort_values('Sharpe', ascending=False)
)

print(sharpe_table.to_string(formatters={
    'Ann_Mean': '{:+.2%}'.format,
    'Ann_Std':  '{:.2%}'.format,
    'Sharpe':   '{:+.3f}'.format,
}))

       Ann_Mean Ann_Std Sharpe
Index                         
创业板指    +18.31%  30.42% +0.602
中证1000   +8.84%  25.28% +0.350
沪深300    +5.79%  16.99% +0.341
上证50     +3.28%  15.38% +0.213


In [4]:
stock_files = {
    # HS300 sample (5), large-caps
    '寒武纪':     'data/prices/sh_688256_2023-04-17_2026-04-17_qfq.csv',
    '国泰君安':   'data/prices/sh_601211_2023-04-17_2026-04-17_qfq.csv',
    '中金公司':   'data/prices/sh_601995_2023-04-17_2026-04-17_qfq.csv',
    '新希望':     'data/prices/sz_000876_2023-04-17_2026-04-17_qfq.csv',
    '中天科技':   'data/prices/sh_600522_2023-04-17_2026-04-17_qfq.csv',
    # ZZ1000 sample (5), small-to-mid-caps
    '华曙高科':   'data/prices/sh_688433_2023-04-17_2026-04-17_qfq.csv',
    '长亮科技':   'data/prices/sz_300348_2023-04-17_2026-04-17_qfq.csv',
    '京新药业':   'data/prices/sz_002020_2023-04-17_2026-04-17_qfq.csv',
    '安徽合力':   'data/prices/sh_600761_2023-04-17_2026-04-17_qfq.csv',
    '承德露露':   'data/prices/sz_000848_2023-04-17_2026-04-17_qfq.csv',
}

hs300_names  = ['寒武纪', '国泰君安', '中金公司', '新希望', '中天科技']
zz1000_names = ['华曙高科', '长亮科技', '京新药业', '安徽合力', '承德露露']

stock_returns = {}
for label, path in stock_files.items():
    df = pd.read_csv(path, parse_dates=['date']).set_index('date').sort_index()
    close = pd.to_numeric(df['close'], errors='coerce')
    returns = close.pct_change().dropna()
    stock_returns[label] = returns
    print(f"{label:8s}: {len(returns):4d} rows, "
          f"{returns.index.min().date()} → {returns.index.max().date()}")

hs300_matrix  = pd.DataFrame({n: stock_returns[n] for n in hs300_names}).dropna()
zz1000_matrix = pd.DataFrame({n: stock_returns[n] for n in zz1000_names}).dropna()
basket_hs300  = hs300_matrix.mean(axis=1)
basket_zz1000 = zz1000_matrix.mean(axis=1)

print(f"\nHS300 basket:  {len(basket_hs300)} rows")
print(f"ZZ1000 basket: {len(basket_zz1000)} rows")

寒武纪     :  726 rows, 2023-04-18 → 2026-04-17
国泰君安    :  726 rows, 2023-04-18 → 2026-04-17
中金公司    :  726 rows, 2023-04-18 → 2026-04-17
新希望     :  726 rows, 2023-04-18 → 2026-04-17
中天科技    :  726 rows, 2023-04-18 → 2026-04-17
华曙高科    :  726 rows, 2023-04-18 → 2026-04-17
长亮科技    :  726 rows, 2023-04-18 → 2026-04-17
京新药业    :  726 rows, 2023-04-18 → 2026-04-17
安徽合力    :  726 rows, 2023-04-18 → 2026-04-17
承德露露    :  726 rows, 2023-04-18 → 2026-04-17

HS300 basket:  726 rows
ZZ1000 basket: 726 rows


In [5]:
def build_sharpe_table(returns_dict):
    rows = []
    for label, returns in returns_dict.items():
        ann_mean = returns.mean() * TRADING_DAYS_A_SHARE
        ann_std  = returns.std()  * np.sqrt(TRADING_DAYS_A_SHARE)
        sharpe   = compute_sharpe(returns, rf_annual=0.0)
        rows.append({
            'Label':    label,
            'Ann_Mean': ann_mean,
            'Ann_Std':  ann_std,
            'Sharpe':   sharpe,
        })
    return (pd.DataFrame(rows)
              .set_index('Label')
              .sort_values('Sharpe', ascending=False))


all_series = dict(stock_returns)
all_series['[Basket] HS300']  = basket_hs300
all_series['[Basket] ZZ1000'] = basket_zz1000

results = build_sharpe_table(all_series)

print(results.to_string(formatters={
    'Ann_Mean': '{:+.2%}'.format,
    'Ann_Std':  '{:.2%}'.format,
    'Sharpe':   '{:+.3f}'.format,
}))

                Ann_Mean Ann_Std Sharpe
Label                                  
寒武纪              +90.63%  74.18% +1.222
[Basket] HS300   +23.00%  25.90% +0.888
华曙高科             +52.85%  64.28% +0.822
中天科技             +27.36%  38.43% +0.712
[Basket] ZZ1000  +18.14%  29.75% +0.610
国泰君安              +9.48%  24.32% +0.390
长亮科技             +15.67%  56.10% +0.279
承德露露              +6.39%  25.73% +0.248
京新药业              +8.46%  36.77% +0.230
安徽合力              +7.32%  39.75% +0.184
中金公司              -1.50%  32.36% -0.046
新希望              -10.98%  24.94% -0.440


In [6]:
execute_smoke_tests()

drawdown smoke test passed
sharpe smoke test passed
sortino smoke test passed
limit detection smoke test: OK
All smoke tests passed.


In [7]:
def build_ratio_table(returns_dict):
    rows = []
    for label, returns in returns_dict.items():
        ann_mean = returns.mean() * TRADING_DAYS_A_SHARE
        ann_std  = returns.std()  * np.sqrt(TRADING_DAYS_A_SHARE)
        sharpe   = compute_sharpe(returns, rf_annual=0.0)
        sortino  = compute_sortino(returns, target_annual=0.0)
        rows.append({
            'Label':    label,
            'Ann_Mean': ann_mean,
            'Ann_Std':  ann_std,
            'Sharpe':   sharpe,
            'Sortino':  sortino,
            'Ratio':    sortino / sharpe if sharpe > 0 else np.nan,
        })
    return (pd.DataFrame(rows)
              .set_index('Label')
              .sort_values('Sortino', ascending=False))


all_series = dict(stock_returns)
all_series['[Basket] HS300']  = basket_hs300
all_series['[Basket] ZZ1000'] = basket_zz1000
all_series.update(index_returns)

results = build_ratio_table(all_series)

print(results.to_string(formatters={
    'Ann_Mean': '{:+.2%}'.format,
    'Ann_Std':  '{:.2%}'.format,
    'Sharpe':   '{:+.3f}'.format,
    'Sortino':  '{:+.3f}'.format,
    'Ratio':    '{:.2f}'.format,
}))

                Ann_Mean Ann_Std Sharpe Sortino Ratio
Label                                                
寒武纪              +90.63%  74.18% +1.222  +2.097  1.72
[Basket] HS300   +23.00%  25.90% +0.888  +1.398  1.57
华曙高科             +52.85%  64.28% +0.822  +1.319  1.60
中天科技             +27.36%  38.43% +0.712  +1.120  1.57
创业板指             +18.31%  30.42% +0.602  +0.985  1.64
[Basket] ZZ1000  +18.14%  29.75% +0.610  +0.907  1.49
国泰君安              +9.48%  24.32% +0.390  +0.624  1.60
沪深300             +5.79%  16.99% +0.341  +0.514  1.51
中证1000            +8.84%  25.28% +0.350  +0.500  1.43
长亮科技             +15.67%  56.10% +0.279  +0.431  1.54
承德露露              +6.39%  25.73% +0.248  +0.388  1.56
京新药业              +8.46%  36.77% +0.230  +0.359  1.56
上证50              +3.28%  15.38% +0.213  +0.320  1.50
安徽合力              +7.32%  39.75% +0.184  +0.278  1.51
中金公司              -1.50%  32.36% -0.046  -0.071   NaN
新希望              -10.98%  24.94% -0.440  -0.645   NaN
